In [ ]:
!pip install weaviate-client

In [ ]:
import weaviate

client = weaviate.connect_to_local()

In [ ]:
# Delete the collection if it exists
if client.collections.exists("traffic_detection"):
    client.collections.delete("traffic_detection")
    print("Deleted 'traffic_detection' collection")
else:
    print("Collection 'traffic_detection' does not exist")


In [ ]:
import weaviate.classes.config as wc

# Set up index with vector configuration
# In Weaviate v4, use wc.Configure instead of client.Configure
index = client.collections.create(
    name="traffic_detection",
    properties=[
        wc.Property(
            name="track_id",
            data_type=wc.DataType.INT
        ),
        wc.Property(
            name="video_id",
            data_type=wc.DataType.INT,
        ),
        wc.Property(
            name="timestamp",
            data_type=wc.DataType.INT,
        ),
        wc.Property(
            name="frame_number",
            data_type=wc.DataType.INT,
        ),
    ],
    vectorizer_config=wc.Configure.Vectorizer.none(),  # Or use a specific vectorizer if needed
    vector_index_config=wc.Configure.VectorIndex.hnsw(
        distance_metric=wc.VectorDistances.COSINE,
        ef_construction=128,
        max_connections=64,
        # Removed PQ quantizer for accurate distance calculations
        # Use quantizer=wc.Configure.VectorIndex.Quantizer.pq() for production with large datasets
    )
)

In [ ]:
from weaviate.classes.query import Filter

cars_collection = client.collections.get("traffic_detection")

# Method 1: Delete all objects by using a filter that matches everything
result = cars_collection.data.delete_many(
    where=Filter.by_property("video_id").not_equal(0)
)

print(f"Deleted {result.matches} objects (successful: {result.successful}, failed: {result.failed})")

In [ ]:
import torch

# Properties (metadata) for each record
car_data = [
    {
        "video_id": 1,
        "timestamp": 102,
        "frame_number": 10024,
        "track_id": 0,
    },
    {
        "video_id": 1,
        "timestamp": 102,
        "frame_number": 10024,
        "track_id": 1,
    },
    {
        "video_id": 1,
        "timestamp": 102,
        "frame_number": 10024,
        "track_id": 2,
    }
]

# Vectors (embeddings) for each record - kept separate!
vectors = [
    torch.ones(128).tolist(),           # All 1.0s
    torch.zeros(128).tolist(),          # All 0.0s
    (0 - torch.ones(128)).tolist(),   # All 0.8s
]

# Get the collection object
cars_collection = client.collections.get("traffic_detection")

# Insert the documents with vectors SEPARATE from properties
for car, vec in zip(car_data, vectors):
    cars_collection.data.insert(
        properties=car,  # Metadata only (no vector here!)
        vector=vec       # Vector passed separately for similarity search!
    )

print("Inserted car data into the database.")


In [ ]:
# print number of items in cars_collecion
count = cars_collection.aggregate.over_all(total_count=True)
print(f"Number of items in cars_collection: {count.total_count}")

# print also embeddings
# print all items with their embeddings in cars_collection
all_cars = cars_collection.query.fetch_objects(limit=100)  # adjust limit as needed
for idx, car in enumerate(all_cars.objects):
    vector = car.properties.get("vector")
    print(f"Car {idx+1}: {car.properties},\n  Embedding (first 5 dims): {vector[:5] if vector else None}\n")


In [ ]:
from weaviate.classes.query import MetadataQuery

# Query vector: all 0.9s (should be closest to 0.8s, then 1.0s, furthest from 0.0s)
query_vector = (torch.ones(128)).tolist()
query_vector[:64] = torch.zeros(64)

response = cars_collection.query.near_vector(
    near_vector=query_vector,
    limit=3,
    return_metadata=MetadataQuery(distance=True),  # Request distance in results
    include_vector=True
)

# Display results with distance
print(f"Query vector: all values = 0.9\n")
for i, obj in enumerate(response.objects, 1):
    print(obj.properties)
    print(obj.metadata.distance)
    print(torch.tensor(obj.vector["default"]))

In [ ]:
list(response.objects)